# How to Group Data in Pandas DataFrames with `groupby()`:

This guide demonstrates how to use the Pandas `groupby()` function to aggregate and analyze meteorological data.  The `groupby()` function is essential for computing statistics by categories, such as averaging temperatures by station, calculating monthly precipitation totals, or analyzing seasonal patterns.

## Sample Dataset Setup

First, let's create a comprehensive weather dataset:

In [ ]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Create sample meteorological data for one year
dates = pd.date_range('2025-01-01', periods=365, freq='D')
stations = ['KMSP', 'KORD', 'KJFK', 'KSEA', 'KDFW']

# Create data for multiple stations
data_list = []
for station in stations:
    # Different climate characteristics for each station
    base_temp = {'KMSP': 8, 'KORD': 10, 'KJFK':  12, 'KSEA':  11, 'KDFW': 18}[station]
    base_precip = {'KMSP': 2, 'KORD': 2.5, 'KJFK': 3, 'KSEA': 4, 'KDFW': 2}[station]
    
    for i, date in enumerate(dates):
        # Seasonal temperature variation
        temp = base_temp + 15 * np.sin((i - 80) * 2 * np.pi / 365) + np.random.normal(0, 3)
        
        data_list.append({
            'date': date,
            'station': station,
            'temperature': temp,
            'humidity': np.clip(60 + np.random.normal(0, 15), 20, 100),
            'pressure': 1013 + np.random.normal(0, 8),
            'wind_speed': np.abs(np.random.normal(5, 3)),
            'precipitation': np.random.exponential(base_precip) if np.random.random() > 0.7 else 0,
            'cloud_cover': np.random.choice(['Clear', 'Partly Cloudy', 'Cloudy', 'Overcast']),
            'wind_direction': np.random.choice(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])
        })

df = pd.DataFrame(data_list)

# Add derived columns
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()
df['season'] = df['month'].map({12: 'Winter', 1: 'Winter', 2: 'Winter',
                                 3: 'Spring', 4: 'Spring', 5: 'Spring',
                                 6: 'Summer', 7: 'Summer', 8: 'Summer',
                                 9: 'Fall', 10: 'Fall', 11: 'Fall'})
df['day_of_week'] = df['date'].dt.day_name()

# Display the first few rows
print("Dataset shape:", df.shape)
print("\nStations:", df['station'].unique())
print("\nDate range:", df['date'].min(), "to", df['date'].max())

# Show the first 10 rows of the DataFrame
df.head(10)

Dataset shape: (1825, 13)

Stations: ['KMSP' 'KORD' 'KJFK' 'KSEA' 'KDFW']

Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00


,date,station,temperature,humidity,pressure,wind_speed,precipitation,cloud_cover,wind_direction,month,month_name,season,day_of_week
0,2025-01-01,KMSP,-5.229455,57.926035,1018.181508,9.569090,0.000000,Cloudy,W,1,January,Winter,Wednesday
1,2025-01-02,KMSP,-1.930087,71.511521,1009.244205,6.627680,0.000000,Clear,N,1,January,Winter,Thursday
2,2025-01-03,KMSP,-5.885620,31.300796,999.200657,3.313137,0.000000,Partly Cloudy,NW,1,January,Winter,Friday
3,2025-01-04,KMSP,-9.275031,38.815444,1024.725190,4.322671,0.000000,Cloudy,SE,1,January,Winter,Saturday
4,2025-01-05,KMSP,-8.119247,61.663839,1003.792051,6.127094,0.000000,Overcast,N,1,January,Winter,Sunday
5,2025-01-06,KMSP,-8.218863,55.624594,1008.186347,10.556835,0.000000,Clear,NW,1,January,Winter,Monday
6,2025-01-07,KMSP,-3.875888,41.687345,1014.670909,0.879010,0.000000,Partly Cloudy,S,1,January,Winter,Tuesday
7,2025-01-08,KMSP,-4.050448,62.570524,1012.074814,4.096689,0.000000,Partly Cloudy,W,1,January,Winter,Wednesday
8,2025-01-09,KMSP,-7.565862,75.856833,1015.748946,0.289120,0.000000,Cloudy,S,1,January,Winter,Thursday
9,2025-01-10,KMSP,-8.128607,69.175144,1021.247996,7.793840,0.153859,Cloudy,NE,1,January,Winter,Friday


## Basic GroupBy Examples

### 1. Group by Single Column - Average by Station
Calculate the mean temperature at each weather station:

In [4]:
# Group by station and calculate mean temperature
station_avg_temp = df.groupby('station')['temperature'].mean()

print("Average Temperature by Station:")
print("="*50)
print(station_avg_temp.round(2))
print(f"\nHottest station: {station_avg_temp.idxmax()} ({station_avg_temp.max():.2f}°C)")
print(f"Coldest station: {station_avg_temp.idxmin()} ({station_avg_temp.min():.2f}°C)")

Average Temperature by Station:
station
KDFW    17.66
KJFK    11.89
KMSP     7.98
KORD     9.93
KSEA    10.82
Name: temperature, dtype: float64

Hottest station: KDFW (17.66°C)
Coldest station: KMSP (7.98°C)


### 2. Multiple Aggregations on One Column
Calculate various statistics for wind speed by station:

In [5]:
# Multiple statistics for wind speed
wind_stats = df.groupby('station')['wind_speed'].agg(['mean', 'min', 'max', 'std'])

print("Wind Speed Statistics by Station (m/s):")
print("="*50)
print(wind_stats.round(2))

Wind Speed Statistics by Station (m/s):
         mean   min    max   std
station                         
KDFW     5.04  0.00  13.06  2.78
KJFK     4.95  0.01  13.36  2.78
KMSP     5.64  0.04  12.90  2.86
KORD     5.24  0.02  13.61  2.81
KSEA     5.01  0.02  12.89  2.85


### 3. Group Multiple Columns
Calculate statistics for multiple variables:

In [6]:
# Average multiple columns by station
station_summary = df.groupby('station')[['temperature', 'humidity', 'pressure']].mean()

print("Average Conditions by Station:")
print("="*50)
print(station_summary.round(2))

Average Conditions by Station:
         temperature  humidity  pressure
station                                 
KDFW           17.66     60.06   1014.11
KJFK           11.89     59.27   1013.11
KMSP            7.98     59.50   1013.07
KORD            9.93     60.41   1012.60
KSEA           10.82     59.55   1013.29


### 4. Counting Observations
Count the number of observations per group:

In [7]:
# Count observations by cloud cover type
cloud_counts = df.groupby('cloud_cover').size()

print("Observations by Cloud Cover:")
print("="*50)
print(cloud_counts.sort_values(ascending=False))
print(f"\nMost common:  {cloud_counts.idxmax()} ({cloud_counts.max()} days)")

Observations by Cloud Cover:
cloud_cover
Partly Cloudy    477
Clear            475
Overcast         469
Cloudy           404
dtype: int64

Most common:  Partly Cloudy (477 days)


## Intermediate Examples

### 5. Group by Multiple Columns
Analyze data by station and season:

In [8]:
# Average temperature by station and season
seasonal_temps = df.groupby(['station', 'season'])['temperature'].mean().round(2)

print("Average Temperature by Station and Season (°C):")
print("="*50)
print(seasonal_temps)

Average Temperature by Station and Season (°C):
station  season
KDFW     Fall      12.24
         Spring    23.45
         Summer    29.45
         Winter     5.18
KJFK     Fall       6.48
         Spring    17.15
         Summer    24.62
         Winter    -1.05
KMSP     Fall       2.29
         Spring    13.54
         Summer    20.26
         Winter    -4.49
KORD     Fall       4.21
         Spring    15.51
         Summer    22.33
         Winter    -2.64
KSEA     Fall       4.92
         Spring    16.24
         Summer    23.62
         Winter    -1.82
Name: temperature, dtype: float64


In [9]:
# Unstack to create a more readable table
seasonal_temps_table = df.groupby(['station', 'season'])['temperature'].mean().unstack()

print("\nTemperature by Station and Season (Table Format):")
print("="*50)
print(seasonal_temps_table.round(2))


Temperature by Station and Season (Table Format):
season    Fall  Spring  Summer  Winter
station                               
KDFW     12.24   23.45   29.45    5.18
KJFK      6.48   17.15   24.62   -1.05
KMSP      2.29   13.54   20.26   -4.49
KORD      4.21   15.51   22.33   -2.64
KSEA      4.92   16.24   23.62   -1.82


### 6. Custom Aggregation Functions
Use custom functions with groupby:

In [10]:
# Calculate temperature range (max - min) for each station
def temp_range(x):
    return x.max() - x.min()

station_temp_range = df.groupby('station')['temperature'].agg(temp_range)

print("Temperature Range by Station (°C):")
print("="*50)
print(station_temp_range.round(2).sort_values(ascending=False))

Temperature Range by Station (°C):
station
KMSP    45.36
KSEA    44.26
KJFK    43.09
KDFW    41.69
KORD    41.49
Name: temperature, dtype: float64


### 7. Different Aggregations for Different Columns
Apply different functions to different columns:

In [11]:
# Different aggregations for different columns
station_stats = df.groupby('station').agg({
    'temperature': ['mean', 'max', 'min'],
    'precipitation': 'sum',
    'wind_speed': 'mean',
    'humidity': 'mean'
})

print("Comprehensive Station Statistics:")
print("="*50)
print(station_stats.round(2))

Comprehensive Station Statistics:
        temperature               precipitation wind_speed humidity
               mean    max    min           sum       mean     mean
station                                                            
KDFW          17.66  38.53  -3.16        239.14       5.04    60.06
KJFK          11.89  33.63  -9.45        294.37       4.95    59.27
KMSP           7.98  29.44 -15.93        226.91       5.64    59.50
KORD           9.93  29.99 -11.49        193.50       5.24    60.41
KSEA          10.82  34.77  -9.49        400.98       5.01    59.55


### 8. Named Aggregations
Create more readable column names with named aggregations:

In [12]:
# Named aggregations for clarity
station_summary = df.groupby('station').agg(
    avg_temp=('temperature', 'mean'),
    max_temp=('temperature', 'max'),
    min_temp=('temperature', 'min'),
    total_precip=('precipitation', 'sum'),
    avg_wind=('wind_speed', 'mean'),
    observations=('temperature', 'count')
)

print("Station Summary with Named Aggregations:")
print("="*50)
print(station_summary. round(2))

Station Summary with Named Aggregations:
         avg_temp  max_temp  min_temp  total_precip  avg_wind  observations
station                                                                    
KDFW        17.66     38.53     -3.16        239.14      5.04           365
KJFK        11.89     33.63     -9.45        294.37      4.95           365
KMSP         7.98     29.44    -15.93        226.91      5.64           365
KORD         9.93     29.99    -11.49        193.50      5.24           365
KSEA        10.82     34.77     -9.49        400.98      5.01           365


### 9. Filtering Groups
Filter groups based on conditions:

In [ ]:
# Find stations with average temperature above 12°C
warm_stations = df.groupby('station')['temperature'].mean()
warm_stations = warm_stations[warm_stations > 12]

print("Stations with Average Temperature > 12°C:")
print("="*50)
print(warm_stations.round(2))

Stations with Average Temperature > 12°C:
station
KDFW    17.66
Name: temperature, dtype: float64


### 10. Group by Time Period
Analyze monthly patterns:

In [14]:
# Monthly average temperature across all stations
monthly_temp = df.groupby('month_name')['temperature'].mean()

# Reorder by month
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
monthly_temp = monthly_temp. reindex(month_order)

print("Average Temperature by Month (°C):")
print("="*50)
print(monthly_temp.round(2))

Average Temperature by Month (°C):
month_name
January      -2.12
February      2.93
March        10.41
April        17.52
May          23.63
June         26.79
July         25.24
August       20.22
September    12.98
October       5.34
November     -0.22
December     -3.33
Name: temperature, dtype: float64


## Advanced Examples

### 11. Transform - Keep Original Shape
Add group statistics back to the original dataframe:

In [15]:
# Add station average temperature to each row
df['station_avg_temp'] = df.groupby('station')['temperature'].transform('mean')

# Calculate deviation from station average
df['temp_deviation'] = df['temperature'] - df['station_avg_temp']

print("Sample of data with station averages and deviations:")
print("="*50)
print(df[['date', 'station', 'temperature', 'station_avg_temp', 'temp_deviation']].head(10).round(2))

Sample of data with station averages and deviations:
        date station  temperature  station_avg_temp  temp_deviation
0 2025-01-01    KMSP        -5.23              7.98          -13.21
1 2025-01-02    KMSP        -1.93              7.98           -9.91
2 2025-01-03    KMSP        -5.89              7.98          -13.87
3 2025-01-04    KMSP        -9.28              7.98          -17.26
4 2025-01-05    KMSP        -8.12              7.98          -16.10
5 2025-01-06    KMSP        -8.22              7.98          -16.20
6 2025-01-07    KMSP        -3.88              7.98          -11.86
7 2025-01-08    KMSP        -4.05              7.98          -12.03
8 2025-01-09    KMSP        -7.57              7.98          -15.55
9 2025-01-10    KMSP        -8.13              7.98          -16.11


### 12. Apply Custom Functions
Use apply() with groupby for complex operations:

In [16]:
# Find the hottest day at each station
def hottest_day(group):
    idx = group['temperature'].idxmax()
    return pd.Series({
        'date': group. loc[idx, 'date'],
        'temperature': group.loc[idx, 'temperature'],
        'humidity': group.loc[idx, 'humidity']
    })

hottest_days = df.groupby('station').apply(hottest_day)

print("Hottest Day at Each Station:")
print("="*50)
print(hottest_days.round(2))

Hottest Day at Each Station:
              date  temperature  humidity
station                                  
KDFW    2025-06-24        38.53     77.86
KJFK    2025-06-20        33.63     65.87
KMSP    2025-07-10        29.44     59.14
KORD    2025-06-28        29.99     47.02
KSEA    2025-06-16        34.77     70.87


/var/folders/51/b9llyv5s4dd7llx0xjt1rzkw0000gr/T/ipykernel_51447/3449253209.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hottest_days = df.groupby('station').apply(hottest_day)


### 13. Percentile Calculations
Calculate percentiles within groups:

In [17]:
# Calculate temperature percentiles by station
def percentiles(x):
    return pd.Series({
        '10th':  x.quantile(0.10),
        '25th': x.quantile(0.25),
        '50th':  x.quantile(0.50),
        '75th': x.quantile(0.75),
        '90th':  x.quantile(0.90)
    })

temp_percentiles = df.groupby('station')['temperature'].apply(percentiles).unstack()

print("Temperature Percentiles by Station (°C):")
print("="*50)
print(temp_percentiles.round(2))

Temperature Percentiles by Station (°C):
         10th  25th   50th   75th   90th
station                                 
KDFW     3.37  7.61  17.61  27.33  31.84
KJFK    -3.10  1.85  12.12  22.33  26.25
KMSP    -6.29 -2.29   7.95  17.65  22.98
KORD    -4.70  0.01   9.74  20.13  24.47
KSEA    -4.14  1.03  10.44  20.76  26.08


### 14. Rolling Calculations within Groups
Calculate rolling averages for each station:

In [18]:
# Sort by station and date first
df_sorted = df.sort_values(['station', 'date'])

# Calculate 7-day rolling average temperature for each station
df_sorted['temp_7day_avg'] = df_sorted.groupby('station')['temperature'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

print("Sample with 7-day rolling average:")
print("="*50)
print(df_sorted[['date', 'station', 'temperature', 'temp_7day_avg']].head(15).round(2))

Sample with 7-day rolling average:
           date station  temperature  temp_7day_avg
1460 2025-01-01    KDFW         0.50           0.50
1461 2025-01-02    KDFW         0.34           0.42
1462 2025-01-03    KDFW         2.62           1.15
1463 2025-01-04    KDFW         2.03           1.37
1464 2025-01-05    KDFW         0.65           1.23
1465 2025-01-06    KDFW         4.75           1.81
1466 2025-01-07    KDFW         2.35           1.89
1467 2025-01-08    KDFW         6.56           2.76
1468 2025-01-09    KDFW         1.72           2.95
1469 2025-01-10    KDFW        -3.16           2.13
1470 2025-01-11    KDFW         1.83           2.10
1471 2025-01-12    KDFW         1.48           2.22
1472 2025-01-13    KDFW         7.96           2.68
1473 2025-01-14    KDFW         1.50           2.56
1474 2025-01-15    KDFW         5.42           2.39


### 15. Cumulative Calculations
Calculate cumulative values within groups:

In [19]:
# Cumulative precipitation by station
df_sorted['cumulative_precip'] = df_sorted.groupby('station')['precipitation'].cumsum()

# Show year-end totals
year_end_precip = df_sorted.groupby('station')['cumulative_precip'].last()

print("Total Annual Precipitation by Station (mm):")
print("="*50)
print(year_end_precip.round(2).sort_values(ascending=False))

Total Annual Precipitation by Station (mm):
station
KSEA    400.98
KJFK    294.37
KDFW    239.14
KMSP    226.91
KORD    193.50
Name: cumulative_precip, dtype: float64


### 16. Pivot Tables
Create summary tables with pivot_table (advanced groupby):

In [20]:
# Create pivot table:  stations vs seasons
pivot = df.pivot_table(
    values='temperature',
    index='station',
    columns='season',
    aggfunc='mean'
)

# Reorder seasons
pivot = pivot[['Winter', 'Spring', 'Summer', 'Fall']]

print("Temperature Pivot Table - Station vs Season (°C):")
print("="*50)
print(pivot.round(2))

Temperature Pivot Table - Station vs Season (°C):
season   Winter  Spring  Summer   Fall
station                               
KDFW       5.18   23.45   29.45  12.24
KJFK      -1.05   17.15   24.62   6.48
KMSP      -4.49   13.54   20.26   2.29
KORD      -2.64   15.51   22.33   4.21
KSEA      -1.82   16.24   23.62   4.92


### 17. Multiple Aggregations with Pivot Table
Show multiple statistics in a pivot table:

In [21]:
# Pivot with multiple aggregation functions
pivot_multi = df.pivot_table(
    values='precipitation',
    index='station',
    columns='season',
    aggfunc=['sum', 'mean', 'count']
)

print("Precipitation Pivot Table - Multiple Statistics: ")
print("="*50)
print(pivot_multi.round(2))

Precipitation Pivot Table - Multiple Statistics: 
            sum                         mean                      count  \
season     Fall  Spring Summer  Winter  Fall Spring Summer Winter  Fall   
station                                                                   
KDFW      50.07   61.30  65.39   62.39  0.55   0.67   0.71   0.69    91   
KJFK      66.74   31.58  95.87  100.18  0.73   0.34   1.04   1.11    91   
KMSP      72.77   52.92  38.85   62.37  0.80   0.58   0.42   0.69    91   
KORD      43.24   71.97  39.59   38.70  0.48   0.78   0.43   0.43    91   
KSEA     102.33  101.41  90.00  107.24  1.12   1.10   0.98   1.19    91   

                              
season  Spring Summer Winter  
station                       
KDFW        92     92     90  
KJFK        92     92     90  
KMSP        92     92     90  
KORD        92     92     90  
KSEA        92     92     90  


## Meteorological Use Cases

### 18. Precipitation Days Analysis
Count rainy days by station and season:

In [22]:
# Create rainy day indicator
df['rainy_day'] = (df['precipitation'] > 0.1).astype(int)

# Count rainy days by station and season
rainy_days = df.groupby(['station', 'season'])['rainy_day'].sum().unstack()
rainy_days = rainy_days[['Winter', 'Spring', 'Summer', 'Fall']]

print("Rainy Days by Station and Season:")
print("="*50)
print(rainy_days)

Rainy Days by Station and Season:
season   Winter  Spring  Summer  Fall
station                              
KDFW         26      22      33    29
KJFK         35      13      27    23
KMSP         26      26      27    33
KORD         26      20      23    26
KSEA         21      23      31    28


### 19. Extreme Weather Events
Identify and count extreme conditions:

In [23]:
# Define extreme conditions
df['extreme_heat'] = (df['temperature'] > 30).astype(int)
df['extreme_cold'] = (df['temperature'] < 0).astype(int)
df['high_wind'] = (df['wind_speed'] > 15).astype(int)

# Count extreme events by station
extreme_events = df.groupby('station').agg({
    'extreme_heat': 'sum',
    'extreme_cold': 'sum',
    'high_wind': 'sum'
})

extreme_events. columns = ['Heat Days (>30°C)', 'Cold Days (<0°C)', 'High Wind Days (>15 m/s)']

print("Extreme Weather Events by Station:")
print("="*50)
print(extreme_events)

Extreme Weather Events by Station:
         Heat Days (>30°C)  Cold Days (<0°C)  High Wind Days (>15 m/s)
station                                                               
KDFW                    57                 7                         0
KJFK                     7                71                         0
KMSP                     0               119                         0
KORD                     0                91                         0
KSEA                     7                85                         0


### 20. Wind Direction Frequency
Analyze prevailing wind directions by station:

In [24]:
# Count wind direction frequency by station
wind_freq = df.groupby(['station', 'wind_direction']).size().unstack(fill_value=0)

# Calculate percentages
wind_freq_pct = wind_freq.div(wind_freq.sum(axis=1), axis=0) * 100

print("Wind Direction Frequency by Station (%):")
print("="*50)
print(wind_freq_pct.round(1))

Wind Direction Frequency by Station (%):
wind_direction     E     N    NE    NW     S    SE    SW     W
station                                                       
KDFW            12.3  12.9  10.7  11.2  14.0  11.0  11.8  16.2
KJFK            10.7  11.8  11.8  13.2  15.9  11.0  12.6  13.2
KMSP            10.7  12.3   7.9  11.5  17.5  13.4  13.4  13.2
KORD            11.5  10.4  14.5  10.7  14.8  12.6  13.4  12.1
KSEA            12.1   8.8  14.5  15.6  12.6  12.1  12.9  11.5


In [25]:
# Find most common wind direction for each station
prevailing_wind = wind_freq.idxmax(axis=1)

print("\nPrevailing Wind Direction by Station:")
print("="*50)
print(prevailing_wind)


Prevailing Wind Direction by Station:
station
KDFW     W
KJFK     S
KMSP     S
KORD     S
KSEA    NW
dtype: object


### 21. Temperature Variability Analysis
Identify stations with most variable weather:

In [27]:
# Calculate temperature statistics by station
temp_variability = df.groupby('station')['temperature'].agg([
    ('mean', 'mean'),
    ('std', 'std'),
    ('range', lambda x: x.max() - x.min()),
    ('cv', lambda x: x.std() / x.mean() * 100)  # Coefficient of variation
])

temp_variability. columns = ['Mean Temp (°C)', 'Std Dev', 'Range (°C)', 'Coef. of Variation (%)']

print("Temperature Variability by Station:")
print("="*50)
print(temp_variability.round(2))
print(f"\nMost variable: {temp_variability['Coef. of Variation (%)'].idxmax()}")

Temperature Variability by Station:
         Mean Temp (°C)  Std Dev  Range (°C)  Coef. of Variation (%)
station                                                             
KDFW              17.66    10.96       41.69                   62.05
KJFK              11.89    11.20       43.09                   94.23
KMSP               7.98    11.12       45.36                  139.33
KORD               9.93    11.10       41.49                  111.77
KSEA              10.82    11.21       44.26                  103.60

Most variable: KMSP


### 22. Seasonal Precipitation Patterns
Compare wet and dry seasons:

In [28]:
# Total and average precipitation by season and station
seasonal_precip = df.groupby(['season', 'station'])['precipitation'].agg(['sum', 'mean', 'count'])

print("Seasonal Precipitation Statistics:")
print("="*50)
print(seasonal_precip.round(2))

Seasonal Precipitation Statistics:
                   sum  mean  count
season station                     
Fall   KDFW      50.07  0.55     91
       KJFK      66.74  0.73     91
       KMSP      72.77  0.80     91
       KORD      43.24  0.48     91
       KSEA     102.33  1.12     91
Spring KDFW      61.30  0.67     92
       KJFK      31.58  0.34     92
       KMSP      52.92  0.58     92
       KORD      71.97  0.78     92
       KSEA     101.41  1.10     92
Summer KDFW      65.39  0.71     92
       KJFK      95.87  1.04     92
       KMSP      38.85  0.42     92
       KORD      39.59  0.43     92
       KSEA      90.00  0.98     92
Winter KDFW      62.39  0.69     90
       KJFK     100.18  1.11     90
       KMSP      62.37  0.69     90
       KORD      38.70  0.43     90
       KSEA     107.24  1.19     90


### 23. Climate Zones Classification
Group and classify stations by temperature characteristics:

In [29]:
# Calculate climate metrics
climate_metrics = df.groupby('station').agg({
    'temperature': ['mean', 'min', 'max'],
    'precipitation': 'sum'
}).round(2)

climate_metrics.columns = ['Avg Temp', 'Min Temp', 'Max Temp', 'Annual Precip']

# Classify by average temperature
climate_metrics['Climate Zone'] = pd.cut(
    climate_metrics['Avg Temp'],
    bins=[-np.inf, 10, 15, np.inf],
    labels=['Cool', 'Moderate', 'Warm']
)

print("Climate Classification by Station:")
print("="*50)
print(climate_metrics)

Climate Classification by Station:
         Avg Temp  Min Temp  Max Temp  Annual Precip Climate Zone
station                                                          
KDFW        17.66     -3.16     38.53         239.14         Warm
KJFK        11.89     -9.45     33.63         294.37     Moderate
KMSP         7.98    -15.93     29.44         226.91         Cool
KORD         9.93    -11.49     29.99         193.50         Cool
KSEA        10.82     -9.49     34.77         400.98     Moderate


### 24. Growing Degree Days
Calculate agricultural metrics:

In [30]:
# Calculate growing degree days (GDD) - base temperature 10°C
df['gdd'] = df['temperature'].apply(lambda x: max(0, x - 10))

# Sum by station and season
gdd_summary = df.groupby(['station', 'season'])['gdd'].sum().unstack()
gdd_summary = gdd_summary[['Spring', 'Summer', 'Fall', 'Winter']]

print("Growing Degree Days by Station and Season:")
print("="*50)
print(gdd_summary. round(0))

Growing Degree Days by Station and Season:
season   Spring  Summer   Fall  Winter
station                               
KDFW     1238.0  1790.0  361.0    21.0
KJFK      704.0  1345.0  113.0     0.0
KMSP      466.0   944.0   36.0     0.0
KORD      571.0  1134.0   75.0     0.0
KSEA      623.0  1253.0   71.0     0.0


## Comparison:  Different GroupBy Approaches

### Method 1: Basic aggregation

In [31]:
# Simple mean
result1 = df.groupby('station')['temperature'].mean()
print("Method 1 - Basic mean:")
print(result1.round(2))

Method 1 - Basic mean:
station
KDFW    17.66
KJFK    11.89
KMSP     7.98
KORD     9.93
KSEA    10.82
Name: temperature, dtype: float64


### Method 2: Using agg()

In [32]:
# Using agg with function name
result2 = df.groupby('station')['temperature'].agg('mean')
print("\nMethod 2 - Using agg():")
print(result2.round(2))


Method 2 - Using agg():
station
KDFW    17.66
KJFK    11.89
KMSP     7.98
KORD     9.93
KSEA    10.82
Name: temperature, dtype: float64


### Method 3: Named aggregation

In [33]:
# Named aggregation (most readable for complex queries)
result3 = df.groupby('station').agg(
    avg_temperature=('temperature', 'mean')
)
print("\nMethod 3 - Named aggregation:")
print(result3.round(2))


Method 3 - Named aggregation:
         avg_temperature
station                 
KDFW               17.66
KJFK               11.89
KMSP                7.98
KORD                9.93
KSEA               10.82


## Key Concepts Summary

### Main GroupBy Methods:
- `.mean()`, `.sum()`, `.count()`, `.min()`, `.max()`, `.std()` - Direct aggregations
- `.agg()` - Apply one or more functions
- `.transform()` - Keep original shape of data
- `.apply()` - Apply custom functions
- `.filter()` - Filter groups based on conditions
- `.size()` - Count items in each group

### Common Patterns:
1. **Single column, single function**: `df.groupby('col1')['col2'].mean()`
2. **Multiple columns, single function**: `df.groupby('col1')[['col2', 'col3']].mean()`
3. **Single column, multiple functions**: `df.groupby('col1')['col2'].agg(['mean', 'sum'])`
4. **Multiple columns, different functions**: `df.groupby('col1').agg({'col2': 'mean', 'col3': 'sum'})`
5. **Multiple grouping columns**: `df.groupby(['col1', 'col2'])['col3'].mean()`

## Practice Exercises

Try these on your own in the cells below:

### Exercise 1
Calculate the average humidity for each season across all stations.

In [ ]:
# Your code here


### Exercise 2
Find the station with the highest total annual precipitation.

In [ ]:
# Your code here


### Exercise 3
Create a pivot table showing average pressure by station (rows) and cloud cover type (columns).

In [ ]:
# Your code here


### Exercise 4
For each station, find the month with the highest average temperature.

In [ ]:
# Your code here


### Exercise 5
Calculate the percentage of days with precipitation > 5mm for each station.

In [ ]:
# Your code here


## Quick Reference Table

Here's a summary of common groupby operations:

In [34]:
# Create a reference table
reference = pd.DataFrame({
    'Operation': [
        'Group and average',
        'Group and sum',
        'Group and count',
        'Multiple aggregations',
        'Different agg per column',
        'Custom function',
        'Transform',
        'Filter groups'
    ],
    'Syntax': [
        "df.groupby('col')['value'].mean()",
        "df.groupby('col')['value'].sum()",
        "df.groupby('col').size()",
        "df.groupby('col')['value'].agg(['mean', 'sum'])",
        "df.groupby('col').agg({'val1': 'mean', 'val2': 'sum'})",
        "df.groupby('col')['value'].agg(lambda x: custom_func(x))",
        "df.groupby('col')['value'].transform('mean')",
        "df.groupby('col').filter(lambda x: x['value']. mean() > 10)"
    ]
})

print("GroupBy Quick Reference:")
print("="*80)
reference

GroupBy Quick Reference:


,Operation,Syntax
0,Group and average,df.groupby('col')['value'].mean()
1,Group and sum,df.groupby('col')['value'].sum()
2,Group and count,df.groupby('col').size()
3,Multiple aggregations,"df.groupby('col')['value'].agg(['mean', 'sum'])"
4,Different agg per column,"df.groupby('col').agg({'val1': 'mean', 'val2':..."
5,Custom function,df.groupby('col')['value'].agg(lambda x: custo...
6,Transform,df.groupby('col')['value'].transform('mean')
7,Filter groups,df.groupby('col').filter(lambda x: x['value']....


## Resources

- [Pandas GroupBy Documentation](https://pandas.pydata.org/docs/reference/groupby.html)
- [Pandas Aggregation Guide](https://pandas.pydata.org/docs/user_guide/groupby.html)

---

Happy data aggregating! 📊🌡️⛈️